In [1]:
import pyomo.environ as pyo
import pyomo.dae as dae
import math
import random
from pyomo.opt import SolverStatus, TerminationCondition

In [ ]:
def build_joint_pid_model_dae(
    scenarios: dict,     # {sid: {"prob":..., "Ku":..., "tau":..., "d":[T+1], "sp":[T+1]}}
    T=30, h=0.1,
    weights=(1.0, 0.01),
    bounds={"x":(-20,20), "u":(-20,20), "Kp":(0,100), "Ki":(0,1000), "Kd":(0,1000)},
):
    """
    多场景联合最优（确定性等价）：DAE建模 + BACKWARD离散。
      dx/dt = (-x + Ku*u + d)/tau
      e = sp - x
      dI/dt = e
      u = Kp*e + Ki*I + Kd*de/dt
      J = sum_s prob_s * ∫ (w_e e^2 + w_u u^2) dt
    """
    horizon = T * h
    w_e, w_u = weights

    m = pyo.ConcreteModel()
    m.t = dae.ContinuousSet(bounds=(0.0, horizon))

    # 共享的一阶段增益
    m.Kp = pyo.Var(bounds=bounds["Kp"])
    m.Ki = pyo.Var(bounds=bounds["Ki"])
    m.Kd = pyo.Var(bounds=bounds["Kd"])

    sids = sorted(scenarios.keys())
    m.S = pyo.Set(initialize=sids)
    m.B = pyo.Block(m.S)

    for s in sids:
        data = scenarios[s]
        prob, Ku, tau = data["prob"], data["Ku"], data["tau"]
        d_seq, sp_seq = data["d"], data["sp"]
        assert len(d_seq) == T+1 and len(sp_seq) == T+1

        b = m.B[s]
        # 轨迹变量
        b.x = pyo.Var(m.t, bounds=bounds["x"])
        b.e = pyo.Var(m.t)
        b.u = pyo.Var(m.t, bounds=bounds["u"])
        b.I = pyo.Var(m.t)

        # 导数
        b.dxdt = dae.DerivativeVar(b.x, wrt=m.t)
        b.dedt = dae.DerivativeVar(b.e, wrt=m.t)
        b.dIdt = dae.DerivativeVar(b.I, wrt=m.t)

        # 外部信号
        b.d_s  = pyo.Param(m.t, initialize=0.0, mutable=True)
        b.sp_s = pyo.Param(m.t, initialize=0.0, mutable=True)

        # 约束
        @b.Constraint(m.t)
        def e_def(bb, t):   return bb.e[t] == bb.sp_s[t] - bb.x[t]

        @b.Constraint(m.t)
        def x_dyn(bb, t):   return bb.dxdt[t] == (-bb.x[t] + Ku*bb.u[t] + bb.d_s[t]) / tau

        @b.Constraint(m.t)
        def I_dyn(bb, t):   return bb.dIdt[t] == bb.e[t]

        @b.Constraint(m.t)
        def pid_law(bb, t): return bb.u[t] == m.Kp*bb.e[t] + m.Ki*bb.I[t] + m.Kd*bb.dedt[t]

        # 初值
        b.x0 = pyo.Constraint(expr=b.x[m.t.first()] == 0.0)
        b.I0 = pyo.Constraint(expr=b.I[m.t.first()] == 0.0)

        # 场景积分成本
        def integrand(bb, t):
            return w_e*bb.e[t]**2 + w_u*bb.u[t]**2
        b.cost_int = dae.Integral(m.t, wrt=m.t, rule=integrand)
        b.prob = prob

    # 期望目标
    m.obj = pyo.Objective(
        expr=sum(m.B[s].prob * m.B[s].cost_int for s in m.S),
        sense=pyo.minimize
    )

    # DAE 离散化（BACKWARD）
    disc = pyo.TransformationFactory('dae.finite_difference')
    disc.apply_to(m, nfe=T, wrt=m.t, scheme='BACKWARD')

    # 在离散网格点赋值 d, sp
    grid = list(m.t)  # 共 T+1 个点
    for s in sids:
        b = m.B[s]
        d_seq, sp_seq = scenarios[s]["d"], scenarios[s]["sp"]
        assert len(grid) == len(d_seq) == len(sp_seq)
        for i, t in enumerate(grid):
            b.d_s[t]  = float(d_seq[i])
            b.sp_s[t] = float(sp_seq[i])

    return m

def solve_joint_local_dae(m):
    opt = pyo.SolverFactory('ipopt')
    # 根据需要调选项
    opt.options.update({'tol':1e-8, 'acceptable_tol':1e-6})
    res = opt.solve(m, tee=False)
    ok = (res.solver.status == SolverStatus.ok) and \
         (res.solver.termination_condition in [TerminationCondition.optimal,
                                               TerminationCondition.locallyOptimal])
    return ok

def multistart_joint_dae(model_builder, scenarios, T, h, weights, bounds,
                         n_starts=20, seed=0):
    random.seed(seed)
    best_val = float('inf')
    best_K   = None
    for k in range(n_starts):
        m = model_builder(scenarios, T=T, h=h, weights=weights, bounds=bounds)
        # 随机初始化共享增益（在有界域内）
        for var, (lb, ub) in [(m.Kp, bounds["Kp"]), (m.Ki, bounds["Ki"]), (m.Kd, bounds["Kd"])]:
            var.set_value(lb + (ub-lb)*random.random())
        if solve_joint_local_dae(m):
            val = pyo.value(m.obj)
            if val < best_val:
                best_val = val
                best_K = (pyo.value(m.Kp), pyo.value(m.Ki), pyo.value(m.Kd))
    return best_val, best_K

# ====== 示例调用 ======
if __name__ == "__main__":
    T, h = 30, 0.1
    times = [t for t in range(T+1)]
    def step_sp(t): return 1.0 if t*h >= 0.5 else 0.0
    def make_d(amp): return [amp*math.sin(0.5*t*h) for t in times]

    scenarios = {
        1: {"prob": 0.4, "Ku": 3.0, "tau": 2.0, "d": make_d(0.2), "sp": [step_sp(t) for t in times]},
        # 可加更多场景：
        # 2: {"prob": 0.4, "Ku": 2.7, "tau": 1.8, "d": make_d(0.5), "sp": [step_sp(t) for t in times]},
        # 3: {"prob": 0.2, "Ku": 3.3, "tau": 2.2, "d": make_d(0.8), "sp": [step_sp(t) for t in times]},
    }
    bounds={"x":(-20,20), "u":(-20,20), "Kp":(0,100), "Ki":(0,1000), "Kd":(0,1000)}
    weights=(1.0, 0.01)

    # 单次（局部）解
    m = build_joint_pid_model_dae(scenarios, T=T, h=h, weights=weights, bounds=bounds)
    ok = solve_joint_local_dae(m)
    if ok:
        print("[DAE-Local] objective =", pyo.value(m.obj),
              "K* =", (pyo.value(m.Kp), pyo.value(m.Ki), pyo.value(m.Kd)))

    # 多起点（更接近全局）
    best_val, best_K = multistart_joint_dae(
        build_joint_pid_model_dae, scenarios, T, h, weights, bounds, n_starts=30, seed=42
    )
    print("[DAE-Multistart] best objective =", best_val, "best K* =", best_K)


ERROR: Rule failed for Integral 'B[1].cost_int' with index None: TypeError:
'NoneType' object is not iterable
ERROR: Constructing component 'B[1].cost_int' from data=None failed:
        TypeError: 'NoneType' object is not iterable


TypeError: 'NoneType' object is not iterable